# 🧱 TP IA — Treino YOLOv8 para Deteção de Peças LEGO

**Dataset:** Lego Detection v3 (Roboflow)  
**Classes:** 40 tipos de peças LEGO  
**Modelo:** YOLOv8s (small — equilíbrio velocidade/precisão)

## Pré-requisitos
```bash
pip install ultralytics roboflow opencv-python tensorboard
```

## 1. Verificar ambiente

In [ ]:
import ultralytics
from ultralytics import YOLO
import torch
import os

ultralytics.checks()

# Detectar o melhor dispositivo disponível
if torch.cuda.is_available():
    DEVICE = 0  # GPU NVIDIA
    print(f"✅ GPU NVIDIA: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    DEVICE = 'mps'  # Apple Silicon
    print("✅ Apple Silicon MPS")
else:
    DEVICE = 'cpu'
    print("⚠️  CPU (treino será mais lento)")

print(f"Dispositivo selecionado: {DEVICE}")

## 2. Descarregar dataset do Roboflow

**Opção A** — Descarregar via API do Roboflow (recomendado)  
**Opção B** — Usar o ZIP já descarregado manualmente

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# OPÇÃO A — Descarregar via Roboflow API
# Descomenta este bloco se quiseres descarregar diretamente
# ─────────────────────────────────────────────────────────────────────

# import roboflow
#
# roboflow.login()  # Faz login na primeira vez
#
# # A tua API key do Roboflow (encontras em roboflow.com → Account → Roboflow API)
# API_KEY = "COLOCA_AQUI_A_TUA_API_KEY"
#
# rf = roboflow.Roboflow(api_key=API_KEY)
# project = rf.workspace("nunos-workspace-rwjb5").project("lego-detection-sqibp")
# dataset = project.version(3).download("yolov8", location="../dataset_yolo")
#
# DATASET_YAML = dataset.location + "/data.yaml"
# print(f"Dataset guardado em: {dataset.location}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────
# OPÇÃO B — Usar o ZIP já descarregado do Roboflow
# Coloca o .zip na pasta dataset_yolo/ e corre esta célula
# ─────────────────────────────────────────────────────────────────────

import zipfile
import os
from pathlib import Path

ZIP_PATH = "../dataset_yolo/Lego_detection_v3-test-model-22-05_yolov8.zip"
EXTRACT_DIR = "../dataset_yolo/lego_v3"

if not os.path.exists(EXTRACT_DIR):
    print(f"A extrair dataset para {EXTRACT_DIR}...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print("✅ Extração concluída!")
else:
    print("✅ Dataset já extraído.")

DATASET_YAML = os.path.abspath(f"{EXTRACT_DIR}/data.yaml")
print(f"data.yaml: {DATASET_YAML}")

## 3. Corrigir paths no data.yaml
O Roboflow usa paths relativos (`../train/images`) que podem não funcionar dependendo de onde corres o treino. Esta célula corrige isso automaticamente.

In [ ]:
import yaml

dataset_dir = Path(DATASET_YAML).parent

with open(DATASET_YAML, 'r') as f:
    data = yaml.safe_load(f)

print("=== data.yaml original ===")
print(f"  train: {data.get('train')}")
print(f"  val:   {data.get('val')}")
print(f"  test:  {data.get('test')}")
print(f"  nc:    {data.get('nc')}")
print(f"  names: {data.get('names')}")

# Converter para paths absolutos
fixed_yaml_path = str(dataset_dir / "data_fixed.yaml")

data['train'] = str((dataset_dir / 'train' / 'images').resolve())
data['val']   = str((dataset_dir / 'valid' / 'images').resolve())
data['test']  = str((dataset_dir / 'test'  / 'images').resolve())

with open(fixed_yaml_path, 'w') as f:
    yaml.dump(data, f, allow_unicode=True, default_flow_style=False)

print("\n=== data_fixed.yaml (paths absolutos) ===")
print(f"  train: {data['train']}")
print(f"  val:   {data['val']}")
print(f"  test:  {data['test']}")
print(f"\n✅ Guardado em: {fixed_yaml_path}")

# Confirmar que as pastas existem
for split, path in [('train', data['train']), ('val', data['val']), ('test', data['test'])]:
    exists = os.path.exists(path)
    count  = len(list(Path(path).glob('*.jpg'))) + len(list(Path(path).glob('*.png'))) if exists else 0
    print(f"  {split}: {'✅' if exists else '❌'} {count} imagens")

## 4. Treinar o modelo YOLOv8s

**Estimativas de tempo:**
- CPU: ~8-15 min/epoch → 100 epochs ≈ 14-25 horas ⚠️
- GPU NVIDIA (ex: RTX 3060): ~1-2 min/epoch → 100 epochs ≈ 2-3 horas
- Apple M-series (MPS): ~2-4 min/epoch → 100 epochs ≈ 4-7 horas

💡 **Se só tens CPU**, usa `epochs=30` para testar primeiro.

In [ ]:
from ultralytics import YOLO

# ─── Configuração do treino ───────────────────────────────────────────
# Muda para 'yolov8n.pt' se tiveres pouca memória (nano, mais rápido)
# Muda para 'yolov8m.pt' se quiseres maior precisão e tiveres GPU
MODEL_SIZE = "yolov8s.pt"   # small — recomendado pelo enunciado

# Número de epochs — ajusta conforme o teu hardware
# CPU apenas → começa com 30; GPU → usa 100
EPOCHS = 100

# Batch size — reduz para 8 se tiveres erros de memória
BATCH = 16

# Nome do experimento (aparece na pasta runs/)
EXP_NAME = "lego_v3_s"
# ─────────────────────────────────────────────────────────────────────

model = YOLO(MODEL_SIZE)

results = model.train(
    data=fixed_yaml_path,
    epochs=EPOCHS,
    imgsz=640,
    batch=BATCH,
    device=DEVICE,
    name=EXP_NAME,
    project="../modelos",

    # ── Otimizador ────────────────────────────────────────────────────
    optimizer='auto',       # SGD ou Adam — YOLO escolhe automaticamente
    lr0=0.01,               # Learning rate inicial
    lrf=0.01,               # Learning rate final (lr0 * lrf)
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=3.0,

    # ── Critério de paragem ───────────────────────────────────────────
    patience=50,            # Para se não melhorar em 50 epochs (early stopping)

    # ── Augmentation (já feita no Roboflow, mas estas são em tempo real) ─
    hsv_h=0.015,            # Variação de matiz
    hsv_s=0.7,              # Variação de saturação
    hsv_v=0.4,              # Variação de brilho
    fliplr=0.5,             # Flip horizontal (50% probabilidade)
    flipud=0.0,             # Flip vertical (desativado)
    mosaic=1.0,             # Mosaico 4 imagens (muito útil para LEGO)
    scale=0.5,              # Zoom aleatório ±50%
    translate=0.1,          # Translação aleatória ±10%

    # ── Apple Silicon: desativar AMP se aparecerem NaN losses ─────────
    amp=True,               # Muda para False se vires 'NaN loss' no treino

    # ── Outros ───────────────────────────────────────────────────────
    plots=True,             # Gera gráficos de métricas e exemplos visuais
    save=True,              # Guarda os pesos do modelo
    verbose=True,
)

print("\n✅ Treino concluído!")
print(f"Modelo guardado em: ../modelos/{EXP_NAME}/weights/best.pt")

## 5. Avaliar o modelo no conjunto de teste

In [ ]:
import json

# Caminho para o melhor modelo treinado
BEST_MODEL = f"../modelos/{EXP_NAME}/weights/best.pt"

model = YOLO(BEST_MODEL)

# Avaliar no conjunto de teste
metrics = model.val(
    data=fixed_yaml_path,
    split='test',           # Usa o split de teste (não o de validação)
    device=DEVICE,
    imgsz=640,
    conf=0.25,
    iou=0.5,
    plots=True,
    save_json=True,
    project="../results",
    name=f"{EXP_NAME}_eval",
)

# Resumo das métricas
print("\n" + "="*50)
print("📊 MÉTRICAS NO CONJUNTO DE TESTE")
print("="*50)
print(f"mAP@0.5:      {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Precision:    {metrics.box.mp:.4f}")
print(f"Recall:       {metrics.box.mr:.4f}")

# Guardar métricas em JSON (para o entregável D3)
eval_results = {
    "model": BEST_MODEL,
    "dataset": "lego_detection_v3",
    "split": "test",
    "metrics": {
        "mAP50":     round(float(metrics.box.map50), 4),
        "mAP50-95":  round(float(metrics.box.map),   4),
        "precision": round(float(metrics.box.mp),    4),
        "recall":    round(float(metrics.box.mr),    4),
    },
    "per_class": {}
}

# Métricas por classe
class_names = model.names
print("\n📋 Precisão por classe:")
print(f"{'Classe':<35} {'Precision':>10} {'Recall':>10} {'mAP50':>10}")
print("-" * 70)

for i, (p, r, ap50) in enumerate(zip(
    metrics.box.p, metrics.box.r, metrics.box.ap50
)):
    name = class_names[i]
    print(f"{name:<35} {p:>10.4f} {r:>10.4f} {ap50:>10.4f}")
    eval_results["per_class"][name] = {
        "precision": round(float(p), 4),
        "recall":    round(float(r), 4),
        "mAP50":     round(float(ap50), 4),
    }

# Guardar JSON
os.makedirs("../modelos", exist_ok=True)
eval_path = f"../modelos/{EXP_NAME}_eval.json"
with open(eval_path, 'w') as f:
    json.dump(eval_results, f, indent=2)

print(f"\n✅ Métricas guardadas em: {eval_path}")

## 6. TensorBoard — Visualizar curvas de treino

Abre um terminal e corre:
```bash
tensorboard --logdir ../modelos
```
Depois abre http://localhost:6006 no browser.

In [ ]:
# Mostrar as curvas de treino geradas automaticamente pelo YOLO
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

results_dir = Path(f"../modelos/{EXP_NAME}")

plots_to_show = [
    ("results.png",         "Curvas de treino/validação"),
    ("confusion_matrix.png","Matriz de confusão"),
    ("PR_curve.png",        "Curva Precision-Recall"),
    ("F1_curve.png",        "Curva F1"),
    ("val_batch0_pred.jpg", "Previsões no batch de validação"),
]

for filename, title in plots_to_show:
    img_path = results_dir / filename
    if img_path.exists():
        fig, ax = plt.subplots(figsize=(14, 8))
        img = mpimg.imread(str(img_path))
        ax.imshow(img)
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"⚠️  {filename} ainda não existe — corre o treino primeiro.")

## 7. Inferência rápida — testar imagens

Coloca imagens de teste na pasta `input/` e corre esta célula.

In [ ]:
import json
import cv2
from pathlib import Path
from ultralytics import YOLO

BEST_MODEL = f"../modelos/{EXP_NAME}/weights/best.pt"
INPUT_DIR  = Path("../input")
OUTPUT_DIR = Path("../output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

model = YOLO(BEST_MODEL)

SUPPORTED = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
images = [p for p in INPUT_DIR.iterdir() if p.suffix.lower() in SUPPORTED]

if not images:
    print(f"⚠️  Sem imagens em '{INPUT_DIR}'. Adiciona imagens de teste lá.")
else:
    for img_path in sorted(images):
        print(f"🔍 A processar: {img_path.name}")

        results = model.predict(
            source=str(img_path),
            conf=0.25,
            iou=0.7,
            imgsz=640,
            device=DEVICE,
            verbose=False,
        )

        result = results[0]

        # JSON com detecções
        detections = []
        for box in result.boxes:
            x1, y1, x2, y2 = box.xyxy[0].tolist()
            detections.append({
                "class_id":   int(box.cls),
                "class_name": model.names[int(box.cls)],
                "confidence": round(float(box.conf), 4),
                "bbox": {
                    "x1": round(x1, 2), "y1": round(y1, 2),
                    "x2": round(x2, 2), "y2": round(y2, 2),
                    "width":  round(x2 - x1, 2),
                    "height": round(y2 - y1, 2),
                }
            })

        # Guardar JSON
        json_path = OUTPUT_DIR / f"{img_path.stem}.json"
        with open(json_path, 'w') as f:
            json.dump(detections, f, indent=2)

        # Guardar imagem anotada
        annotated = result.plot(conf=True, labels=True, boxes=True, line_width=2)
        out_img_path = OUTPUT_DIR / img_path.name
        cv2.imwrite(str(out_img_path), annotated)

        print(f"   → {len(detections)} deteção(ões) | img: {out_img_path.name} | json: {json_path.name}")

    print("\n✅ Inferência concluída!")